# PharmaLens AI — Module 15
## Market Access & Institutional Intelligence

Institution → Department → Stakeholder → Formulary → Procurement/Tender → Opportunity → KAM Action → Revenue


In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.append(str(Path.cwd()))
from market_access import *


### 1. Load templates


In [ ]:
institutions = pd.read_csv("institutions_template.csv")
formulary = pd.read_csv("formulary_listing_template.csv")
tenders = pd.read_csv("tenders_template.csv")
stakeholders = pd.read_csv("stakeholders_kam_template.csv")
display(institutions.head())
display(formulary.head())


### 2. Institution prioritization & hospital segmentation


In [ ]:
priority = institution_prioritization(institutions)
display(priority[["Institution_ID","Institution_Name","Potential_Value","Institution_Opportunity_Score","Priority_Tier"]])
segments = hospital_segmentation(institutions)
display(segments[["Institution_Name","Hospital_Segment","Institution_Opportunity_Score"]])


### 3. Formulary / listing opportunity


In [ ]:
listing = listing_opportunity_score(formulary)
display(listing[["Product_Name","Institution_Name","Formulary_Status","Listing_Opportunity_Score"]])


### 4. Tender intelligence


In [ ]:
tender_scores = tender_opportunity_score(tenders)
display(tender_scores[["Tender_ID","Institution_Name","Product_Name","Tender_Opportunity_Score"]])


### 5. Account share & untapped opportunity


In [ ]:
accounts = account_share(institutions)
display(accounts[["Institution_Name","Our_Sales","Estimated_Account_Market","Account_Share_%","Untapped_Opportunity_Value"]])


### 6. Stakeholder / KIM priority


In [ ]:
stakeholder_scores = stakeholder_priority(stakeholders)
display(stakeholder_scores[["Institution_Name","Department","Stakeholder_Name","Role","Stakeholder_Priority_Score","Stakeholder_Priority"]])


### 7. Price / access trade-off


In [ ]:
scenarios = pd.DataFrame([
    {"Scenario":"A","Price":100,"Expected_Volume":10000,"Access_Probability_%":30},
    {"Scenario":"B","Price":90,"Expected_Volume":12000,"Access_Probability_%":50},
    {"Scenario":"C","Price":80,"Expected_Volume":15000,"Access_Probability_%":70},
])
scenarios["Expected_Revenue"] = scenarios.apply(
    lambda r: price_access_tradeoff(r["Price"], r["Expected_Volume"], r["Access_Probability_%"]), axis=1
)
display(scenarios)


### 8. Access-barrier intelligence


In [ ]:
examples = [
    {"Formulary_Status":"Non-listed","Procurement_Status":"Open","Competitor_Contract":False,"Price_Barrier":False},
    {"Formulary_Status":"Listed","Procurement_Status":"Restricted","Competitor_Contract":False,"Price_Barrier":False},
    {"Formulary_Status":"Listed","Procurement_Status":"Open","Competitor_Contract":True,"Price_Barrier":False},
    {"Formulary_Status":"Listed","Procurement_Status":"Open","Competitor_Contract":False,"Price_Barrier":True},
]
barriers = pd.DataFrame(examples)
barriers["Barrier"] = barriers.apply(
    lambda r: classify_access_barrier(r["Formulary_Status"], r["Procurement_Status"],
                                      r["Competitor_Contract"], r["Price_Barrier"]), axis=1
)
display(barriers)


### 9. KAM/KIM account-plan structure


In [ ]:
account_plan = build_account_plan({
    "institution_id": "H001",
    "account_name": "Example Hospital",
    "strategic_goal": "Increase institutional share and secure next tender"
})
account_plan


### 10. Production integration notes


**Recommended production flow**
- Ingest licensed market / sales / hospital / tender / formulary data.
- Normalize institution, product, molecule, department and stakeholder IDs.
- Calculate opportunity scores.
- Push ranked accounts and opportunities into Target Planning and SFE.
- Let AI Copilot explain *why* an account is prioritized and propose next actions.
- Keep evidence/source fields for every external intelligence record.
- Enforce tenant isolation and Row-Level Security in Supabase.
